# Batch YOLO inference on carotid ultrasound images

Runs the trained model (`best_YOLO27Jan2024.pt`) on every `XXX.png` file
(3-digit filenames, e.g. `543.png`) in a given folder.

**No preprocessing** (no crop, no median blur, no CLAHE) is applied here,
since these images are already the right size/format and don't need it.

Inference parameters are kept identical to the original notebook:
`max_det=3, iou=0.1, imgsz=480, conf=0.13, save_txt=True, save_conf=True, save=True`.


## 1. Install dependencies

In [ ]:
!pip install ultralytics


## 2. Imports

In [ ]:
import re
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO


## 3. Configuration

Edit these two paths as needed.

In [ ]:
WEIGHTS_PATH = "best_YOLO27Jan2024.pt"   # path to the trained model weights
INPUT_DIR = "train"                        # folder containing the XXX.png images


## 4. Find all `XXX.png` files (3-digit filenames only)

This matches files like `543.png`, but skips things like `543_gray.png` or `5430.png`.

In [ ]:
THREE_DIGIT_PNG = re.compile(r"^\d{3}\.png$", re.IGNORECASE)

def find_xxx_pngs(folder):
    folder = Path(folder)
    return sorted(
        p for p in folder.iterdir()
        if p.is_file() and THREE_DIGIT_PNG.match(p.name)
    )

image_paths = find_xxx_pngs(INPUT_DIR)
print(f"Found {len(image_paths)} matching image(s):")
for p in image_paths:
    print(" ", p.name)


## 5. Run inference (with result caching)

Runs the model once on all images and caches the detections (boxes,
confidences, classes) to `yolo_detections_cache.json` next to the notebook.
On a kernel restart, if the cache already covers every image in `INPUT_DIR`,
inference is skipped entirely and the cached detections are loaded instead.

Delete `yolo_detections_cache.json` (or edit `INPUT_DIR`/add new images) to
force a re-run.

In [ ]:
import json

DETECTIONS_CACHE_PATH = Path("yolo_detections_cache.json")


def load_cached_detections(cache_path, image_paths):
    """Returns a list of detection dicts aligned with image_paths, or None
    if no usable cache exists (missing file, or missing an image)."""
    if not cache_path.exists():
        return None
    with open(cache_path) as f:
        cached = json.load(f)
    by_name = {c["image"]: c for c in cached}
    if any(p.name not in by_name for p in image_paths):
        return None
    return [by_name[p.name] for p in image_paths]


def save_cached_detections(cache_path, detections):
    with open(cache_path, "w") as f:
        json.dump(detections, f)

## 6. Run inference on all images (or load cached detections)

Same parameters as the original notebook. No preprocessing is applied to the
images before running the model. If a usable cache from a previous run
exists, the model is never loaded and inference never runs.

In [ ]:
detections = load_cached_detections(DETECTIONS_CACHE_PATH, image_paths)

if detections is not None:
    print(f"Loaded cached detections for {len(detections)} image(s) from "
          f"{DETECTIONS_CACHE_PATH} - skipping inference.")
else:
    print("No usable cache found - running YOLO inference...")
    model = YOLO(WEIGHTS_PATH)
    results = model.predict(
        [str(p) for p in image_paths],
        max_det=3,
        iou=0.1,
        imgsz=480,
        conf=0.13,
        save_txt=True,
        save_conf=True,
        save=True,
    )
    detections = [
        {
            "image": p.name,
            "boxes": [box.xyxy[0].tolist() for box in result.boxes],
            "confs": [float(box.conf[0]) for box in result.boxes],
            "clss": [int(box.cls[0]) for box in result.boxes],
        }
        for p, result in zip(image_paths, results)
    ]
    save_cached_detections(DETECTIONS_CACHE_PATH, detections)
    print(f"Saved detections for {len(detections)} image(s) to {DETECTIONS_CACHE_PATH}")

## 7. (Optional) Preview a result

Change the index to look at a different image's output.

In [ ]:
idx = 0
img_path = image_paths[idx]
det = detections[idx]

img = cv2.imread(str(img_path))
for (x1, y1, x2, y2), conf in zip(det["boxes"], det["confs"]):
    cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
    cv2.putText(img, f"{conf:.2f}", (int(x1), max(int(y1) - 5, 0)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)

plt.figure(figsize=(6, 8))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title(img_path.name)
plt.axis("off")
plt.show()

## 8. (Optional) Inspect the raw detections for one image

Boxes, confidence scores, and class ids for a given image, straight from the `results` list (same info as the saved txt labels).

In [ ]:
det = detections[idx]
print(f"Image: {image_paths[idx].name}")
print(f"Detections: {len(det['boxes'])}")
for box, conf, cls in zip(det["boxes"], det["confs"], det["clss"]):
    print(f"  class={cls} conf={conf:.3f} bbox(xyxy)={[round(v, 1) for v in box]}")

## 9. Automatischer Vergleich: TP / FP / FN / TN

Vergleicht die YOLO-Vorhersagen (Boxen + Konfidenz) automatisiert mit den manuellen `XXX_labeled.png`-Masken (weiße Fläche = Plaque).

**Vorgehen:**
1. Jede zusammenhängende weiße Region in der Maske wird per Connected-Components-Analyse in eine Ground-Truth-Box umgewandelt (eine Maske kann mehrere getrennte Plaque-Bereiche enthalten).
2. Vorhergesagte Boxen werden den GT-Boxen zugeordnet — zwei Methoden stehen zur Wahl (`MATCH_METHOD`):
   - **`"center"`** (Default): eine Vorhersage zählt als Treffer, wenn der **Mittelpunkt der predicted Box innerhalb der GT-Box** liegt. Das ist robuster als IoU bei dünnen/gebogenen Plaques, wo die Box deutlich größer als die eigentliche Maske ist.
   - **`"iou"`**: klassisches Box-IoU-Matching mit `IOU_MATCH_THRESHOLD`.
3. Zugeordnete Paare → **TP** (mit Konfidenz und IoU). Nicht zugeordnete Vorhersagen → **FP** (mit Konfidenz). Nicht zugeordnete GT-Boxen → **FN**.
4. **TN** gibt es nur auf Bildebene: kein GT-Plaque und keine Vorhersage im ganzen Bild.

In [ ]:
import numpy as np
import pandas as pd

MATCH_METHOD = "center"     # "center" oder "iou"
IOU_MATCH_THRESHOLD = 0.1   # nur relevant, wenn MATCH_METHOD = "iou"
MIN_MASK_AREA = 20          # kleine Rauschflecken in der Maske ignorieren (Pixel)


def mask_to_boxes(mask_path, min_area=MIN_MASK_AREA):
    """Liest eine binäre Labeled-Maske und liefert eine Liste von
    Bounding-Boxen (x1, y1, x2, y2), eine pro zusammenhängender weißer
    Region."""
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)

    boxes = []
    for label_id in range(1, n_labels):  # 0 = Hintergrund
        area = stats[label_id, cv2.CC_STAT_AREA]
        if area < min_area:
            continue
        x = stats[label_id, cv2.CC_STAT_LEFT]
        y = stats[label_id, cv2.CC_STAT_TOP]
        w = stats[label_id, cv2.CC_STAT_WIDTH]
        h = stats[label_id, cv2.CC_STAT_HEIGHT]
        boxes.append((x, y, x + w, y + h))
    return boxes


def compute_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    inter_x1, inter_y1 = max(ax1, bx1), max(ay1, by1)
    inter_x2, inter_y2 = min(ax2, bx2), min(ay2, by2)
    inter_w, inter_h = max(0, inter_x2 - inter_x1), max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter_area
    return inter_area / union if union > 0 else 0.0


def box_center(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)


def center_in_box(pred_box, gt_box):
    """True, wenn der Mittelpunkt der predicted Box innerhalb der GT-Box liegt."""
    cx, cy = box_center(pred_box)
    x1, y1, x2, y2 = gt_box
    return (x1 <= cx <= x2) and (y1 <= cy <= y2)


def center_distance(pred_box, gt_box):
    px, py = box_center(pred_box)
    gx, gy = box_center(gt_box)
    return ((px - gx) ** 2 + (py - gy) ** 2) ** 0.5


def match_detections(gt_boxes, pred_boxes, pred_confs,
                      method="center", iou_threshold=IOU_MATCH_THRESHOLD):
    """Ordnet Vorhersagen den GT-Boxen zu (Greedy).

    method="center": eine Vorhersage gilt als Treffer, wenn ihr Mittelpunkt
        innerhalb der GT-Box liegt. Bei mehreren Kandidaten wird der mit dem
        geringsten Abstand der Mittelpunkte bevorzugt (unter allen Paaren,
        die das Kriterium erfüllen).
    method="iou": klassisches IoU-Matching mit Schwellenwert.

    Gibt eine Liste von Records zurück: je ein Eintrag pro TP, FP oder FN.
    """
    records = []
    matched_gt = set()
    matched_pred = set()

    pairs = []
    for gi, gt_box in enumerate(gt_boxes):
        for pi, pred_box in enumerate(pred_boxes):
            if method == "center":
                if center_in_box(pred_box, gt_box):
                    # kleinerer Abstand = "besseres" Match -> negativ sortieren
                    score = -center_distance(pred_box, gt_box)
                    pairs.append((score, gi, pi))
            elif method == "iou":
                iou = compute_iou(gt_box, pred_box)
                if iou >= iou_threshold:
                    pairs.append((iou, gi, pi))
            else:
                raise ValueError(f"Unbekannte method: {method}")

    pairs.sort(reverse=True, key=lambda t: t[0])

    for score, gi, pi in pairs:
        if gi in matched_gt or pi in matched_pred:
            continue
        matched_gt.add(gi)
        matched_pred.add(pi)
        iou_val = compute_iou(gt_boxes[gi], pred_boxes[pi])
        records.append({"type": "TP", "confidence": pred_confs[pi], "iou": iou_val})

    for pi in range(len(pred_boxes)):
        if pi not in matched_pred:
            records.append({"type": "FP", "confidence": pred_confs[pi], "iou": None})

    for gi in range(len(gt_boxes)):
        if gi not in matched_gt:
            records.append({"type": "FN", "confidence": None, "iou": None})

    return records


### Auswertung über alle Bilder

In [ ]:
all_records = []

for idx, img_path in enumerate(image_paths):
    stem = img_path.stem
    labeled_path = Path(INPUT_DIR) / f"{stem}_labeled.png"

    gt_boxes = mask_to_boxes(labeled_path)

    det = detections[idx]
    pred_boxes = [tuple(box) for box in det["boxes"]]
    pred_confs = det["confs"]

    if not gt_boxes and not pred_boxes:
        # Bildebene: kein Plaque vorhanden und keiner vorhergesagt
        all_records.append({
            "image": img_path.name, "type": "TN", "confidence": None, "iou": None
        })
        continue

    recs = match_detections(gt_boxes, pred_boxes, pred_confs,
                             method=MATCH_METHOD, iou_threshold=IOU_MATCH_THRESHOLD)
    for r in recs:
        r["image"] = img_path.name
        all_records.append(r)

df = pd.DataFrame(all_records, columns=["image", "type", "confidence", "iou"])
df

### Diagnose: Warum tauchen manche Bilder mehrfach in der Tabelle auf?

`mask_to_boxes()` erzeugt **eine GT-Box pro zusammenhängender weißer Region** in der Maske — nicht eine Box pro Bild. Wenn eine Maske durch Anti-Aliasing, Kompressionsartefakte oder eine dünne fast unterbrochene Stelle im Plaque in mehrere Blobs zerfällt, entstehen mehrere GT-Boxen für dasselbe Bild → mehrere Zeilen in der Tabelle (z. B. eine TP + eine FN für dieselbe Datei, wenn nur ein Blob getroffen wird).

Diese Zelle zeigt, bei welchen Bildern mehr als eine GT-Box gefunden wurde.

In [ ]:
gt_box_counts = {}
for img_path in image_paths:
    stem = img_path.stem
    labeled_path = Path(INPUT_DIR) / f"{stem}_labeled.png"
    gt_box_counts[img_path.name] = len(mask_to_boxes(labeled_path))

multi_gt = {k: v for k, v in gt_box_counts.items() if v > 1}
print(f"{len(multi_gt)} Bild(er) mit mehr als einer GT-Box:")
for name, n in sorted(multi_gt.items(), key=lambda t: -t[1]):
    print(f"  {name}: {n} GT-Boxen")


### Betroffenes Bild visuell prüfen

Zeigt die Maske mit allen gefundenen GT-Boxen eingezeichnet — so siehst du direkt, ob es sich um echte separate Plaques handelt oder um einen durch Rauschen aufgespaltenen einzelnen Blob.

In [ ]:
def show_mask_with_boxes(stem):
    labeled_path = Path(INPUT_DIR) / f"{stem}_labeled.png"
    mask = cv2.imread(str(labeled_path))
    boxes = mask_to_boxes(labeled_path)

    vis = mask.copy()
    for (x1, y1, x2, y2) in boxes:
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 0, 255), 2)

    plt.figure(figsize=(6, 8))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"{stem}_labeled.png — {len(boxes)} GT-Box(en)")
    plt.axis("off")
    plt.show()

show_mask_with_boxes("85")


### Optional: nahe beieinanderliegende Blobs zusammenführen

Falls es sich tatsächlich um denselben Plaque handelt, der nur durch eine dünne Lücke in zwei Regionen zerfällt, kann eine morphologische Closing-Operation die Lücke vor der Connected-Components-Analyse schließen. `MORPH_CLOSE_KERNEL = 0` deaktiviert das (Default).

In [ ]:
MORPH_CLOSE_KERNEL = 0   # z.B. 5 oder 7 zum Schließen kleiner Lücken; 0 = aus


def mask_to_boxes(mask_path, min_area=MIN_MASK_AREA, close_kernel=MORPH_CLOSE_KERNEL):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

    if close_kernel and close_kernel > 0:
        kernel = np.ones((close_kernel, close_kernel), np.uint8)
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)

    boxes = []
    for label_id in range(1, n_labels):
        area = stats[label_id, cv2.CC_STAT_AREA]
        if area < min_area:
            continue
        x = stats[label_id, cv2.CC_STAT_LEFT]
        y = stats[label_id, cv2.CC_STAT_TOP]
        w = stats[label_id, cv2.CC_STAT_WIDTH]
        h = stats[label_id, cv2.CC_STAT_HEIGHT]
        boxes.append((x, y, x + w, y + h))
    return boxes

# Nach Änderung von MORPH_CLOSE_KERNEL: diese Zelle sowie die Auswertungs-
# Schleife (Abschnitt "Auswertung über alle Bilder") erneut ausführen.


In [ ]:
counts = df["type"].value_counts().reindex(["TP", "FP", "FN", "TN"], fill_value=0)
print("Counts:")
print(counts)

tp, fp, fn = counts["TP"], counts["FP"], counts["FN"]
precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else float("nan")

print(f"\nPrecision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-Score:  {f1:.3f}")

print("\nMittlere Konfidenz pro Kategorie:")
print(df.groupby("type")["confidence"].mean())


### Ergebnis pro Bild exportieren (optional)

In [ ]:
df.to_csv("tp_fp_fn_tn_results.csv", index=False)
print("Gespeichert unter tp_fp_fn_tn_results.csv")


## 10. Interaktives manuelles Labeling-Tool

Zeigt jedes Bild mit der gelabelten Maske als **rote Überlagerung
(40% Transparenz)** an, mit den **YOLO-Detektionsboxen (grün)** oben
drauf eingezeichnet, damit sie durch die Maske nicht verdeckt werden. Du
bewertest per Tastatur, ob die YOLO-Detektion korrekt war.

**Tastenbelegung:**

| Taste | Bedeutung | Ergebnis-Code |
|---|---|---|
| `↑` | korrekt detected | `1` |
| `↓` | nicht detected (falsch negativ) | `0` |
| `→` | etwas detected, wo nichts ist (falsch positiv) | `2` |
| `#` | beides: korrektes nicht detected **und** etwas falsches detected | `7` |
| `←` | vorheriges Bild anzeigen (kein Labeln) | – |
| `z` | Bild als "besonders" flaggen — **springt NICHT weiter** | – |

Nach jedem Tastendruck (außer `←`) wird automatisch zum nächsten noch
nicht gelabelten Bild gesprungen, das Ergebnis in
`labeling_results.csv` gespeichert und die Tabelle darunter aktualisiert.

**Voraussetzung:** `ipympl` muss installiert sein, damit Matplotlib-
Grafiken interaktiv im Notebook laufen (Tastatur-Events funktionieren
sonst nicht). Nach der Installation ggf. Kernel neu starten.

In [ ]:
import sys
print("Aktueller Kernel-Interpreter:", sys.executable)

# Installiert ipympl garantiert in die Umgebung, die dieser Kernel nutzt
# (statt in die des Terminals/einer anderen Python-Installation).
!{sys.executable} -m pip install --upgrade ipympl

print()
print("WICHTIG: Jetzt den Kernel KOMPLETT neu starten (nicht nur \'Restart & Run All\'),")
print("danach erst mit der naechsten Zelle (%matplotlib widget) fortfahren.")


**Wichtig:** Nach der Installation von `ipympl` ggf. den Kernel neu
starten, bevor du fortfährst. Danach diese Zelle ausführen, um den
interaktiven Backend zu aktivieren (muss **vor** dem Erzeugen der
Figure passieren).


In [ ]:
%matplotlib widget
import ipywidgets as widgets
from IPython.display import display
import pandas as pd


**Falls `%matplotlib widget` immer noch "not a recognizable backend name" meldet:**
Führe die Diagnose-Zelle darunter aus und prüfe, ob `ipympl` überhaupt
importierbar ist und welche Version installiert ist.

In [ ]:
try:
    import ipympl
    print("ipympl Version:", ipympl.__version__)
except ImportError as e:
    print("ipympl ist in diesem Kernel NICHT importierbar:", e)
    print("-> Installations-Zelle oben nochmal ausfuehren und Kernel neu starten.")

import matplotlib
print("matplotlib Version:", matplotlib.__version__)
print("Verfuegbare Backends:", matplotlib.rcsetup.non_interactive_bk + matplotlib.rcsetup.interactive_bk)


### Fortschritt laden / initialisieren

Liest `labeling_results.csv`, falls vorhanden (Resume), und ergänzt
neue Bilder. `has_label` wird automatisch aus der Maske bestimmt
(ist überhaupt etwas Weißes drauf?) — das musst du nicht manuell
einschätzen.


In [ ]:
RESULTS_CSV = "labeling_results.csv"
OVERLAY_ALPHA = 0.4


def has_any_label(mask_path):
    if not Path(mask_path).exists():
        return "no"
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return "no"
    return "yes" if (mask > 127).any() else "no"


def build_label_df(image_paths, input_dir, results_csv=RESULTS_CSV):
    if Path(results_csv).exists():
        df = pd.read_csv(results_csv, dtype={"image": str})
    else:
        df = pd.DataFrame(columns=["image", "has_label", "result", "flag"])

    existing = set(df["image"])
    new_rows = []
    for p in image_paths:
        if p.name in existing:
            continue
        labeled_path = Path(input_dir) / f"{p.stem}_labeled.png"
        new_rows.append({
            "image": p.name,
            "has_label": has_any_label(labeled_path),
            "result": pd.NA,
            "flag": "no",
        })
    if new_rows:
        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

    df.to_csv(results_csv, index=False)
    return df


label_df = build_label_df(image_paths, INPUT_DIR)
n_done = label_df["result"].notna().sum()
print(f"{n_done} von {len(label_df)} Bildern bereits gelabelt.")


### Das Labeling-Tool

Zeigt pro Bild das YOLO-Ergebnisbild mit roter Masken-Überlagerung
(40%), eine Legende und den verbleibenden Fortschritt. Unter der
Grafik erscheint die live aktualisierte Ergebnistabelle.

**Zum Bedienen: einmal auf die Grafik klicken, damit sie den
Tastatur-Fokus bekommt — danach mit den Pfeiltasten/`#`/`z` labeln.**


In [ ]:
class LabelingTool:
    def __init__(self, image_paths, input_dir, df, detections, results_csv=RESULTS_CSV, alpha=OVERLAY_ALPHA):
        self.paths_by_name = {p.name: p for p in image_paths}
        self.detections_by_name = {p.name: d for p, d in zip(image_paths, detections)}
        self.input_dir = Path(input_dir)
        self.order = [p.name for p in image_paths]
        self.df = df.set_index("image")
        self.results_csv = results_csv
        self.alpha = alpha

        self.table_out = widgets.Output()
        self.fig, self.ax = plt.subplots(figsize=(6, 8))
        self.fig.canvas.mpl_connect("key_press_event", self.on_key)

        display(widgets.VBox([self.fig.canvas, self.table_out]))

        self.pos = self._first_unlabeled_index()
        self.render()

    # --- Hilfsfunktionen -------------------------------------------------

    def _first_unlabeled_index(self):
        for i, name in enumerate(self.order):
            if pd.isna(self.df.loc[name, "result"]):
                return i
        return len(self.order) - 1  # alles bereits gelabelt

    def remaining_count(self):
        return int(self.df["result"].isna().sum())

    def current_name(self):
        return self.order[self.pos]

    def load_overlay(self, name):
        """Baut das kombinierte Bild: Original + rote Masken-Ueberlagerung
        (Ground Truth) + eingezeichnete YOLO-Boxen (gruen), die Boxen werden
        NACH der Maske gezeichnet, damit sie nicht darunter verschwinden."""
        base = cv2.imread(str(self.paths_by_name[name]))
        base = cv2.cvtColor(base, cv2.COLOR_BGR2RGB).astype(np.float32)

        labeled_path = self.input_dir / f"{Path(name).stem}_labeled.png"
        if labeled_path.exists():
            mask = cv2.imread(str(labeled_path), cv2.IMREAD_GRAYSCALE)
            if mask is not None:
                if mask.shape[:2] != base.shape[:2]:
                    mask = cv2.resize(mask, (base.shape[1], base.shape[0]))
                alpha_mask = (mask > 127).astype(np.float32) * self.alpha
                red = np.zeros_like(base)
                red[..., 0] = 255  # reiner Rotkanal
                for c in range(3):
                    base[..., c] = base[..., c] * (1 - alpha_mask) + red[..., c] * alpha_mask

        overlay = base.astype(np.uint8).copy()

        det = self.detections_by_name[name]
        for (x1, y1, x2, y2), conf in zip(det["boxes"], det["confs"]):
            cv2.rectangle(overlay, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
            cv2.putText(overlay, f"{conf:.2f}", (int(x1), max(int(y1) - 5, 0)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)

        return overlay

    # --- Rendering ---------------------------------------------------------

    def render(self):
        name = self.current_name()
        img = self.load_overlay(name)

        self.ax.clear()
        self.ax.imshow(img)
        self.ax.axis("off")

        row = self.df.loc[name]
        result_str = "-" if pd.isna(row["result"]) else str(int(row["result"]))
        remaining = self.remaining_count()
        flag_val = row["flag"]

        title = (f"{name}   |   Ergebnis: {result_str}   |   Flag: {flag_val}\n"
                 f"Verbleibend: {remaining} von {len(self.order)}")
        self.ax.set_title(title, fontsize=10)

        legend = ("\u2191 korrekt [1]    \u2193 nicht detected [0]    "
                  "\u2192 falsch detected [2]    # beides falsch [7]    "
                  "\u2190 zur\u00fcck    z Flag")
        self.ax.text(0.5, -0.04, legend, transform=self.ax.transAxes,
                      ha="center", va="top", fontsize=8)

        self.fig.canvas.draw_idle()
        self.update_table()

    def update_table(self):
        with self.table_out:
            self.table_out.clear_output(wait=True)
            display(self.df.reset_index()[["image", "has_label", "result", "flag"]])

    def save(self):
        self.df.reset_index().to_csv(self.results_csv, index=False)

    # --- Aktionen ------------------------------------------------------

    def set_result(self, value):
        name = self.current_name()
        self.df.loc[name, "result"] = value
        self.save()
        self.advance()

    def advance(self):
        for i in range(self.pos + 1, len(self.order)):
            if pd.isna(self.df.loc[self.order[i], "result"]):
                self.pos = i
                self.render()
                return
        for i in range(len(self.order)):
            if pd.isna(self.df.loc[self.order[i], "result"]):
                self.pos = i
                self.render()
                return
        self.render()  # alle Bilder gelabelt

    def go_back(self):
        if self.pos > 0:
            self.pos -= 1
            self.render()

    def toggle_flag(self):
        name = self.current_name()
        current = self.df.loc[name, "flag"]
        self.df.loc[name, "flag"] = "no" if current == "yes" else "yes"
        self.save()
        self.render()

    def on_key(self, event):
        key = event.key
        if key == "up":
            self.set_result(1)
        elif key == "down":
            self.set_result(0)
        elif key == "right":
            self.set_result(2)
        elif key in ("#", "numbersign"):
            self.set_result(7)
        elif key == "left":
            self.go_back()
        elif key in ("z", "Z"):
            self.toggle_flag()


tool = LabelingTool(image_paths, INPUT_DIR, label_df, detections)

## Nächste Schritte (mit Matthias)
- Ergebnisse des Label Tools
- Besprechen mit Lars bzw. Georgakhis Lab Mitarbeitern
- Supervised Fine Tuning auf Georgakhis Weights  --> Architektur sollte ich haben, sonst Rücksprache
- Model auf UKB, externer Datensatz, Validierung, Performance Stats
- Datensatz Münster von Georgakis (noch mal andere Auflösung?)
- Object Detection <> Binary Classifier --> Worauf eigentlich trainiert? Methodischer Zusammenhang (nur True/False) oder auch hit auf area of plaque
- Loss definieren beim Finetuning